# H21b: Does prescribed layer imbalance reduce Muon's residual advantage over NormSGD?

**Counterpart script:** `experiments/Tier_2_Symmetry_Reparametrization_Tests/H21b_BALANCE_PREDICTS_DIRECTION_VALUE/run_experiment.py`

This notebook is the reporting companion to the script above. It intentionally **imports and uses the script's returned results** rather than re-implementing the experiment loop, so the notebook stays aligned with the executable source.

## Truthful framing

This pair does **not** establish that layer balance *predicts* directional value as a settled result. Instead, it tests the narrower toy hypothesis:

> In this 4-layer deep linear setup, Muon's residual advantage over NormSGD may shrink as the prescribed imbalance parameter $c$ increases.

The current pair compares **Muon vs NormSGD only**. It does **not** include an oracle per-layer learning-rate SGD control, so any claim that this notebook already "fully controls for per-layer LR" would overstate what the implementation actually does.

## Headline metric kept for parity with the script

For each prescribed imbalance value $c$, the script reports

\[
\text{residual advantage}(c)
= \frac{\text{mean final loss of NormSGD}}{\text{mean final loss of Muon}}.
\]

This notebook keeps that headline metric unchanged for parity, while also using the script's raw per-seed losses to show seed-level variation and uncertainty summaries.

## Notebook workflow

1. Locate and import the counterpart script in a notebook-safe way.
2. Log the exact default configuration, deterministic seeds, and estimated workload.
3. Run the script's default experiment.
4. Build a per-$c$ results table from the returned structured data.
5. Plot:
   - residual advantage vs prescribed imbalance,
   - Muon vs NormSGD final losses with seed-level variation,
   - actual initial layer-norm-ratio diagnostics.
6. Report the script's T1/T2 verdicts and conclude plainly whether the observed evidence supports the toy hypothesis.

**Runtime note.** The default run is not instant: learning rates are selected separately for each optimizer and each $c$, so one full run involves many `train()` calls and is closer to a minute-scale job than a quick interactive toy cell.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    def Markdown(text):
        return text
    def display(obj):
        print(obj)

plt.rcParams['figure.figsize'] = (8, 4.8)
plt.rcParams['axes.grid'] = True

In [ ]:
EXPERIMENT_RELATIVE = Path(
    'experiments/Tier_2_Symmetry_Reparametrization_Tests/'
    'H21b_BALANCE_PREDICTS_DIRECTION_VALUE/run_experiment.py'
)

cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents]
SCRIPT_PATH = None
REPO_ROOT = None

for root in search_roots:
    candidate = root / EXPERIMENT_RELATIVE
    if candidate.exists():
        SCRIPT_PATH = candidate.resolve()
        REPO_ROOT = root.resolve()
        break

if SCRIPT_PATH is None:
    local_candidate = cwd / 'run_experiment.py'
    if local_candidate.exists():
        SCRIPT_PATH = local_candidate.resolve()
        REPO_ROOT = SCRIPT_PATH.parents[3]

if SCRIPT_PATH is None or REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the counterpart script. Open the notebook from the repo or adjust this path-resolution cell.'
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

spec = importlib.util.spec_from_file_location('h21b_run_experiment', SCRIPT_PATH)
h21b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h21b)

print(f'Repo root: {REPO_ROOT}')
print(f'Script path: {SCRIPT_PATH}')
print(f'Notebook path recorded by script: {h21b.NOTEBOOK_PATH}')

## Reproducibility, configuration, and workload

The next cell logs the exact default configuration used by the script, the deterministic seeds, and an estimated `train()` call count. The goal is to make the computational cost and reproducibility of the run explicit before executing it.

In [ ]:
config = h21b.get_default_config()
seeds = h21b.get_default_seeds(
    num_seeds=config['num_seeds'],
    seed_base=config['seed_base'],
    seed_stride=config['seed_stride'],
)
estimated_train_calls = h21b.estimate_train_calls(config)

config_lines = [
    '### Default experiment configuration',
    f"- Experiment ID: `{h21b.EXPERIMENT_ID}`",
    f"- Deterministic seeds: `{seeds}`",
    f"- Prescribed imbalance values: `{config['c_values']}`",
    f"- Muon LR grid: `{config['lr_muon']}`",
    f"- NormSGD LR grid: `{config['lr_normsgd']}`",
    f"- Layers / dimension: `{config['n_layers']}` layers of `{config['dim']}x{config['dim']}` matrices",
    f"- Batch size: `{config['batch_size']}`",
    f"- Steps per train run: `{config['num_steps']}`",
    f"- Momentum: `{config['momentum']}`",
    f"- Newton-Schulz iterations: `{config['ns_iters']}`",
    f"- Selection seeds used per LR sweep: `{config['num_selection_seeds']}`",
    f"- Estimated total `train()` calls for one full run: `{estimated_train_calls}`",
    "- Practical caveat: this notebook reproduces the script's default full run. For faster exploratory checks, call the script function with a reduced config in a scratch cell rather than changing the scientific notebook narrative.",
]

display(Markdown('\n'.join(config_lines)))

## Run the counterpart script's experiment

This cell executes the script's default experiment via `run_experiment(...)` and stores the returned structured results for all downstream tables and plots. The notebook therefore reports **exactly what the script computed** rather than a parallel reimplementation.

In [ ]:
start_time = time.time()
results = h21b.run_experiment(config=config, seeds=seeds, verbose=True)
elapsed_seconds = time.time() - start_time

per_c = results['per_c']
summary = results['summary']

print(f'\nNotebook wall-clock time for this run: {elapsed_seconds:.1f} seconds')

## Build derived summaries from the script output

The script's primary behavior is preserved, but it now returns enough structured data for more serious presentation. The next cell assembles seed-level and per-$c$ summaries without changing the script's core metrics.

In [ ]:
def finite_values(values):
    return np.asarray([float(v) for v in values if np.isfinite(v)], dtype=float)


def safe_mean(values):
    arr = finite_values(values)
    return float(np.mean(arr)) if arr.size else float('inf')


def safe_std(values):
    arr = finite_values(values)
    if arr.size <= 1:
        return 0.0 if arr.size == 1 else float('nan')
    return float(np.std(arr, ddof=1))


def ordinal_ranks(values):
    values = np.asarray(values, dtype=float)
    order = np.argsort(values)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(values) + 1, dtype=float)
    return ranks


def spearman_correlation(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2 or np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return float('nan')
    return float(np.corrcoef(ordinal_ranks(x), ordinal_ranks(y))[0, 1])


summary_rows = []
paired_seed_ratios = {}
muon_seed_losses = {}
norm_seed_losses = {}

for entry in per_c:
    c = float(entry['c'])
    muon_losses = [record['loss'] for record in entry['evaluation_losses']['muon']]
    norm_losses = [record['loss'] for record in entry['evaluation_losses']['normsgd']]

    ratios = []
    for mu_loss, norm_loss in zip(muon_losses, norm_losses):
        if np.isfinite(mu_loss) and np.isfinite(norm_loss):
            ratios.append(float(norm_loss / max(mu_loss, 1e-30)))

    paired_seed_ratios[c] = np.asarray(ratios, dtype=float)
    muon_seed_losses[c] = np.asarray(muon_losses, dtype=float)
    norm_seed_losses[c] = np.asarray(norm_losses, dtype=float)

    init_ratio = entry['actual_initial_norm_ratio']
    summary_rows.append({
        'c': c,
        'best_lr_muon': entry['best_lrs']['muon'],
        'best_lr_normsgd': entry['best_lrs']['normsgd'],
        'muon_mean_loss': entry['mean_losses']['muon'],
        'norm_mean_loss': entry['mean_losses']['normsgd'],
        'muon_std_loss': safe_std(muon_losses),
        'norm_std_loss': safe_std(norm_losses),
        'residual_advantage': entry['residual_advantage'],
        'paired_ratio_mean': safe_mean(ratios),
        'paired_ratio_std': safe_std(ratios),
        'muon_finite': entry['evaluation_finite_counts']['muon'],
        'norm_finite': entry['evaluation_finite_counts']['normsgd'],
        'init_ratio_mean': init_ratio['mean'],
        'init_ratio_min': init_ratio['min'],
        'init_ratio_max': init_ratio['max'],
        'label': entry['directional_advantage_label'],
    })

c_grid = np.asarray([row['c'] for row in summary_rows], dtype=float)
headline_advantages = np.asarray([row['residual_advantage'] for row in summary_rows], dtype=float)
muon_means = np.asarray([row['muon_mean_loss'] for row in summary_rows], dtype=float)
norm_means = np.asarray([row['norm_mean_loss'] for row in summary_rows], dtype=float)
muon_stds = np.asarray([row['muon_std_loss'] for row in summary_rows], dtype=float)
norm_stds = np.asarray([row['norm_std_loss'] for row in summary_rows], dtype=float)
init_ratio_means = np.asarray([row['init_ratio_mean'] for row in summary_rows], dtype=float)
init_ratio_mins = np.asarray([row['init_ratio_min'] for row in summary_rows], dtype=float)
init_ratio_maxs = np.asarray([row['init_ratio_max'] for row in summary_rows], dtype=float)

exploratory_spearman = spearman_correlation(
    np.log(c_grid),
    np.log(np.clip(headline_advantages, 1e-10, None)),
)
monotone_nonincreasing = bool(np.all(np.diff(headline_advantages) <= 0))

In [ ]:
def format_number(x, fmt='{:.3g}'):
    if x is None:
        return 'N/A'
    if isinstance(x, float) and not np.isfinite(x):
        return 'inf'
    return fmt.format(x)


lines = [
    '| c | best LR Muon | best LR NormSGD | mean loss Muon | mean loss NormSGD | residual adv | paired seed ratio mean±sd | finite evals M/N | mean init ||W|| ratio | init ratio range | verdict label |',
    '|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|:---|',
]

for row in summary_rows:
    lines.append(
        "| {c:.1f} | {mu_lr} | {norm_lr} | {mu_mean} | {norm_mean} | {adv}x | {paired} | {mfin}/{nfin} | {ratio_mean} | [{ratio_min}, {ratio_max}] | {label} |".format(
            c=row['c'],
            mu_lr=format_number(row['best_lr_muon'], '{:.3e}'),
            norm_lr=format_number(row['best_lr_normsgd'], '{:.3e}'),
            mu_mean=format_number(row['muon_mean_loss'], '{:.3e}'),
            norm_mean=format_number(row['norm_mean_loss'], '{:.3e}'),
            adv=format_number(row['residual_advantage'], '{:.2f}'),
            paired=f"{format_number(row['paired_ratio_mean'], '{:.2f}')} ± {format_number(row['paired_ratio_std'], '{:.2f}')}",
            mfin=row['muon_finite'],
            nfin=row['norm_finite'],
            ratio_mean=format_number(row['init_ratio_mean'], '{:.2f}'),
            ratio_min=format_number(row['init_ratio_min'], '{:.2f}'),
            ratio_max=format_number(row['init_ratio_max'], '{:.2f}'),
            label=row['label'],
        )
    )

display(Markdown('## Per-$c$ results table\n\n' + '\n'.join(lines)))

## Figure 1: Residual advantage vs prescribed imbalance

This is the script's headline view. The blue line shows the **script's primary metric** (ratio of mean losses), while the gray points show paired seed-level loss ratios built from the raw evaluation losses.

In [ ]:
plt.figure()
for c in c_grid:
    ratios = paired_seed_ratios[c]
    if ratios.size:
        plt.scatter(np.full_like(ratios, c), ratios, color='gray', alpha=0.45, s=35)

plt.plot(c_grid, headline_advantages, marker='o', linewidth=2, color='tab:blue', label='script headline metric')
plt.axhline(1.0, color='black', linestyle='--', linewidth=1, label='parity line (1x)')
plt.axhline(2.0, color='tab:red', linestyle=':', linewidth=1.5, label='2x reference')
plt.xscale('log')
plt.xlabel('Prescribed imbalance c')
plt.ylabel('NormSGD mean loss / Muon mean loss')
plt.title('Residual advantage of Muon over NormSGD')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
if monotone_nonincreasing:
    trend_text = 'The headline curve is monotone non-increasing on the tested grid.'
else:
    trend_text = 'The headline curve is **not** monotone non-increasing on the tested grid.'

figure1_lines = [
    '### Interpretation of Figure 1',
    f"- **Observation.** {trend_text} The script headline residual advantage spans from {headline_advantages.min():.2f}x to {headline_advantages.max():.2f}x across the tested grid.",
    "- **Implication.** Any claim that the current implementation already demonstrates a clean monotone decline with increasing imbalance would overstate the evidence; the result has to be judged from the actual plotted values.",
    "- **Caveat.** The headline metric is a ratio of mean losses, not a full statistical model, and it is based on only a small number of seeds and a sparse grid over $c$.",
]

display(Markdown('\n'.join(figure1_lines)))

## Figure 2: Final loss comparison with seed-level variation

The next figure complements the ratio view by plotting the underlying Muon and NormSGD final losses. Means are shown with simple seed-level standard deviation bars, and individual seed outcomes are overlaid as scatter points.

In [ ]:
plt.figure()
muon_x = c_grid * 0.97
norm_x = c_grid * 1.03

plt.errorbar(muon_x, muon_means, yerr=muon_stds, marker='o', linewidth=2, color='tab:blue', capsize=4, label='Muon mean ± sd')
plt.errorbar(norm_x, norm_means, yerr=norm_stds, marker='s', linewidth=2, color='tab:orange', capsize=4, label='NormSGD mean ± sd')

for c in c_grid:
    mu_losses = finite_values(muon_seed_losses[c])
    no_losses = finite_values(norm_seed_losses[c])
    if mu_losses.size:
        plt.scatter(np.full(mu_losses.shape, c * 0.97), mu_losses, color='tab:blue', alpha=0.4, s=28)
    if no_losses.size:
        plt.scatter(np.full(no_losses.shape, c * 1.03), no_losses, color='tab:orange', alpha=0.4, s=28)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Prescribed imbalance c')
plt.ylabel('Final loss (log scale)')
plt.title('Underlying final losses for Muon and NormSGD')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
better_counts = []
for row in summary_rows:
    if row['residual_advantage'] > 1:
        better = 'Muon'
    elif row['residual_advantage'] < 1:
        better = 'NormSGD'
    else:
        better = 'Tie'
    better_counts.append((row['c'], better))

muon_wins = sum(1 for _, better in better_counts if better == 'Muon')
norm_wins = sum(1 for _, better in better_counts if better == 'NormSGD')

display(Markdown(
    '\n'.join([
        '### Interpretation of Figure 2',
        f'- **Observation.** On the default run, Muon has lower mean final loss than NormSGD at {muon_wins} of {len(summary_rows)} tested $c$ values, while NormSGD is lower at {norm_wins}.',
        "- **Implication.** The ratio view is driven by concrete loss differences rather than an abstract post-processing score; when the ratio falls below 1, NormSGD actually outperforms Muon on the evaluated seeds.",
        "- **Caveat.** The uncertainty shown here is only seed-level variation across five deterministic seeds, not a formal confidence interval from a larger replicated study.",
    ])
))

## Figure 3: Actual initial norm-ratio diagnostic

The script also measures the realized spread of layer Frobenius norms at initialization. This diagnostic checks whether the prescribed imbalance parameter $c$ actually produces larger layer-norm disparity in the initialized network.

In [ ]:
plt.figure()
for entry in per_c:
    c = entry['c']
    per_seed_ratios = [record['ratio'] for record in entry['actual_initial_norm_ratio']['per_seed']]
    plt.scatter(np.full(len(per_seed_ratios), c), per_seed_ratios, color='gray', alpha=0.45, s=35)

plt.plot(c_grid, init_ratio_means, marker='o', linewidth=2, color='tab:green', label='mean realized init ratio')
plt.fill_between(c_grid, init_ratio_mins, init_ratio_maxs, color='tab:green', alpha=0.15, label='seed range')
plt.plot(c_grid, c_grid, linestyle='--', color='black', linewidth=1, label='reference y = c')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Prescribed imbalance c')
plt.ylabel('max_l ||W_l||_F / min_l ||W_l||_F at initialization')
plt.title('Realized initial layer-norm imbalance')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown(
    '\n'.join([
        '### Interpretation of Figure 3',
        f"- **Observation.** The realized initial norm-ratio diagnostic ranges from {init_ratio_means.min():.2f} to {init_ratio_means.max():.2f} in mean value across the tested grid.",
        "- **Implication.** The prescribed scaling parameter does create measurable variation in layer-norm imbalance, so the sweep is changing the initialization condition the experiment claims to manipulate.",
        "- **Caveat.** The current script uses this diagnostic descriptively. It does not fit a predictive model on the realized ratios, and the headline tests are still defined in terms of the prescribed grid over $c$.",
    ])
))

## T1/T2 verdicts and hypothesis check

The script preserves the original two summary checks:

- **T1:** Pearson correlation between $\log c$ and $\log(\text{residual advantage})$ is less than `-0.5`.
- **T2:** residual advantage at $c=1$ is more than `3x` the residual advantage at $c=100$.

The next cell reports those verdicts directly from the script output, and also includes an exploratory Spearman correlation for descriptive context.

In [ ]:
pearson_r = summary['pearson_r_log_c_vs_log_residual_advantage']
t1 = summary['t1_negative_correlation_pass']
t2 = summary['t2_c1_adv_gt_3x_c100_pass']
c_thresh = summary['c_threshold_lt_2x']
support = summary['supports_hypothesis']

threshold_text = f'{c_thresh:.1f}' if c_thresh is not None else f'> {c_grid.max():.1f}'
if support is True:
    support_text = 'supports the toy hypothesis'
elif support is False:
    support_text = 'does **not** support the toy hypothesis'
else:
    support_text = 'is inconclusive for the toy hypothesis'

verdict_lines = [
    '### Script verdict summary',
    f"- First tested $c$ with headline residual advantage below 2x: **{threshold_text}**",
    f"- Pearson corr$(\log c, \log \text{{adv}})$: **{pearson_r:.3f}**" if np.isfinite(pearson_r) else "- Pearson correlation: **undefined**",
    f"- Exploratory Spearman corr$(\log c, \log \text{{adv}})$: **{exploratory_spearman:.3f}**" if np.isfinite(exploratory_spearman) else "- Exploratory Spearman correlation: **undefined**",
    f"- T1 verdict: **{'PASS' if t1 else 'FAIL' if t1 is not None else 'N/A'}**",
    f"- T2 verdict: **{'PASS' if t2 else 'FAIL' if t2 is not None else 'N/A'}**",
    f"- Overall plain-language readout: this default run **{support_text}**.",
]

display(Markdown('\n'.join(verdict_lines)))

In [ ]:
if support is True:
    conclusion = (
        "On this default run, the observed metrics are consistent with the toy hypothesis "
        "that increasing prescribed imbalance reduces Muon's residual advantage over NormSGD."
    )
elif support is False:
    conclusion = (
        "On this default run, the observed metrics do **not** support the toy hypothesis "
        "that increasing prescribed imbalance cleanly reduces Muon's residual advantage over NormSGD."
    )
else:
    conclusion = "On this default run, the available metrics are inconclusive for the toy hypothesis."

conclusion_lines = [
    '## Final conclusion',
    conclusion,
    '',
    '### What is justified by the current pair',
    "- The notebook now reproduces the script rather than shadowing it with copied experiment code.",
    "- The results are presented with raw seed-level losses, explicit configuration logging, and visible figures for the headline metric and its underlying components.",
    "- The conclusion is tied to the observed run, not to an assumed monotone story.",
    '',
    '### What is still *not* justified by the current pair',
    "- A strong causal or generally predictive claim that layer balance determines when Muon's direction matters.",
    "- Any statement that an oracle per-layer learning-rate control has already been included here.",
    "- A robust threshold law inferred from this sparse grid and small number of seeds.",
]

display(Markdown('\n'.join(conclusion_lines)))